# Kalman filters and multi-object tracking
Needs: numpy, scipy, matplotlib
## Worked example 1 · Fusing two Gaussian estimates

In [ ]:
import numpy as np

def fuse(mu_a, var_a, mu_b, var_b):
    """Combine two independent Gaussian estimates of the same quantity."""
    k = var_a / (var_a + var_b)          # how much to trust b relative to a
    mu = mu_a + k * (mu_b - mu_a)
    var = (1 - k) * var_a                # = 1 / (1/var_a + 1/var_b)
    return mu, var, k

# camera says the car is 20.0 m away (sigma 2 m), radar says 22.0 m (sigma 1 m)
mu, var, k = fuse(20.0, 2.0**2, 22.0, 1.0**2)
print(f"fused: {mu:.2f} m, sigma {np.sqrt(var):.3f} m, weight on radar {k:.2f}")

## Worked example 2 · 1D Kalman filter

In [ ]:
def kf1d_step(x, P, z, q, r):
    """One cycle of a 1D Kalman filter with a constant-position model."""
    x_pred, P_pred = x, P + q            # predict: the state may have drifted by q
    K = P_pred / (P_pred + r)            # Kalman gain
    x_new = x_pred + K * (z - x_pred)    # update with the innovation z - x_pred
    P_new = (1 - K) * P_pred
    return x_new, P_new, K

x, P = 0.0, 100.0                        # start: no idea where it is (large P)
# measurements of a target at 10 m, sensor sigma 0.5 m
for z in [10.2, 9.7, 10.4]:
    x, P, K = kf1d_step(x, P, z, q=0.01, r=0.25)
    print(f"z = {z:5.1f}  ->  x = {x:6.3f}, sigma = {np.sqrt(P):.3f}, K = {K:.3f}")

## Kalman filter class and constant-velocity model

In [ ]:
class KalmanFilter:
    """Linear Kalman filter: x' = F x + w,  z = H x + v,  w ~ N(0, Q),  v ~ N(0, R)."""
    def __init__(self, F, H, Q, R, x0, P0):
        self.F, self.H, self.Q, self.R = F, H, Q, R
        self.x, self.P = x0.astype(float), P0.astype(float)

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x

    def update(self, z):
        y = z - self.H @ self.x                      # innovation
        S = self.H @ self.P @ self.H.T + self.R      # innovation covariance
        K = self.P @ self.H.T @ np.linalg.inv(S)     # Kalman gain
        self.x = self.x + K @ y
        self.P = (np.eye(len(self.x)) - K @ self.H) @ self.P
        return y, S

def constant_velocity_model(dt, q, r):
    """State [x, y, vx, vy], measurement [x, y]; q: acceleration noise, r: meas. std."""
    F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)
    H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)
    G = np.array([[dt**2 / 2, 0], [0, dt**2 / 2], [dt, 0], [0, dt]])
    Q = q**2 * G @ G.T                               # white-noise acceleration
    R = r**2 * np.eye(2)
    return F, H, Q, R

## Worked example 3 · Tracking a pedestrian

In [ ]:
# a pedestrian in the image (pixels), 30 fps, walking right and then turning down
rng = np.random.default_rng(0)
dt, n = 1 / 30, 150
t = np.arange(n) * dt
vx = np.where(t < 2.5, 120.0, 120.0 * np.clip(1 - (t - 2.5) / 0.6, 0, 1))   # px/s
vy = np.where(t < 2.5, 0.0, 90.0 * np.clip((t - 2.5) / 0.6, 0, 1))
truth = np.c_[200 + np.cumsum(vx) * dt, 300 + np.cumsum(vy) * dt]

meas = truth + rng.normal(0, 4.0, truth.shape)          # detector noise: 4 px
missed = rng.random(n) < 0.15                            # 15 % missed detections
missed[95:110] = True                                    # plus a short occlusion

F, H, Q, R = constant_velocity_model(dt, q=150.0, r=4.0)
kf = KalmanFilter(F, H, Q, R, x0=np.r_[meas[0], 0, 0], P0=np.diag([16, 16, 1e4, 1e4]))
est = [kf.x[:2].copy()]
for k in range(1, n):
    kf.predict()
    if not missed[k]:
        kf.update(meas[k])
    est.append(kf.x[:2].copy())
est = np.array(est)

ok = ~missed
rmse_raw = np.sqrt(np.mean(np.sum((meas[ok] - truth[ok]) ** 2, axis=1)))
rmse_kf = np.sqrt(np.mean(np.sum((est[ok] - truth[ok]) ** 2, axis=1)))
gap = slice(95, 110)
gap_err = np.sqrt(np.mean(np.sum((est[gap] - truth[gap]) ** 2, axis=1)))
print(f"position error: raw detections {rmse_raw:.2f} px, Kalman filter {rmse_kf:.2f} px")
print(f"during the 0.5 s occlusion (prediction only): {gap_err:.2f} px")
print(f"velocity at the end: ({kf.x[2]:.1f}, {kf.x[3]:.1f}) px/s, true (0.0, 90.0)")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(truth[:, 0], truth[:, 1], "k--", label="truth")
plt.plot(meas[~missed, 0], meas[~missed, 1], ".", ms=3, label="detections")
plt.plot(est[:, 0], est[:, 1], lw=2, label="Kalman estimate")
plt.gca().invert_yaxis(); plt.axis("equal"); plt.legend(); plt.show()

## Worked example 4 · Tuning q and checking NIS

In [ ]:
def run_filter(q, r_assumed=4.0):
    F, H, Q, R = constant_velocity_model(dt, q=q, r=r_assumed)
    kf = KalmanFilter(F, H, Q, R, x0=np.r_[meas[0], 0, 0], P0=np.diag([16, 16, 1e4, 1e4]))
    out, nis = [kf.x[:2].copy()], []
    for k in range(1, n):
        kf.predict()
        if not missed[k]:
            y, S_ = kf.update(meas[k])
            nis.append(y @ np.linalg.inv(S_) @ y)    # normalised innovation squared
        out.append(kf.x[:2].copy())
    e = np.array(out)[ok] - truth[ok]
    return np.sqrt(np.mean(np.sum(e**2, axis=1))), np.mean(nis[10:])

for q in [5, 30, 150, 600, 3000]:
    rmse, nis = run_filter(q)
    print(f"q = {q:5d} px/s^2: RMSE {rmse:5.2f} px, mean NIS {nis:5.2f}  (ideal: 2.0)")

## EKF update with a radar measurement

In [ ]:
# EKF: a radar measures range and bearing of a car in the vehicle frame (x forward, y left)
def h_radar(x):
    px, py = x[0], x[1]
    return np.array([np.hypot(px, py), np.arctan2(py, px)])

def H_radar(x):
    """Jacobian of h at the current estimate: the linearisation step of the EKF."""
    px, py = x[0], x[1]
    r2 = px**2 + py**2
    r = np.sqrt(r2)
    return np.array([[px / r, py / r, 0, 0],
                     [-py / r2, px / r2, 0, 0]])

# predicted state: 20 m ahead, 5 m left
x = np.array([20.0, 5.0, -3.0, 0.0])
P = np.diag([1.0, 1.0, 4.0, 4.0])
R_radar = np.diag([0.3**2, np.radians(1.0)**2])        # 0.3 m range, 1 deg bearing noise
z = np.array([21.0, np.radians(15.5)])                 # measurement

Hj = H_radar(x)
y = z - h_radar(x)                                     # innovation uses the NON-linear h
y[1] = (y[1] + np.pi) % (2 * np.pi) - np.pi            # wrap the angle difference
S_ = Hj @ P @ Hj.T + R_radar
K = P @ Hj.T @ np.linalg.inv(S_)
x_new = x + K @ y
print("predicted position:", x[:2], " updated:", x_new[:2].round(3))
print("range / bearing of update:", h_radar(x_new).round(4), " measured:", z.round(4))

## Simulated MOT scene and detector

In [ ]:
# ---------- a synthetic MOT scenario with ground truth ----------
W_IMG, H_IMG, N_FRAMES = 1280, 720, 300
rng = np.random.default_rng(7)
# x-range of an occluding pillar in front of the scene
PILLAR = (600, 700)

def make_objects(n_obj=12):
    objs = []
    for i in range(n_obj):
        start = rng.integers(0, 200)
        life = rng.integers(120, 260)
        left = rng.random() < 0.5
        w, h = rng.uniform(40, 90), rng.uniform(90, 180)
        x0 = -w / 2 if left else W_IMG + w / 2
        vx = rng.uniform(4, 8) * (1 if left else -1)          # px per frame
        y0 = rng.uniform(150, 560)
        vy = rng.uniform(-0.6, 0.6)
        objs.append(dict(id=i, start=start, end=start + life,
                         x=x0, y=y0, vx=vx, vy=vy, w=w, h=h))
    return objs

def simulate(objs):
    """Return per-frame ground truth and detections (x1, y1, x2, y2, score)."""
    gt, dets = [], []
    state = {o["id"]: dict(o) for o in objs}
    for f in range(N_FRAMES):
        g, d = [], []
        for o in state.values():
            if not (o["start"] <= f < o["end"]):
                continue
            o["vx"] += rng.normal(0, 0.05); o["vy"] += rng.normal(0, 0.05)
            o["x"] += o["vx"]; o["y"] += o["vy"]
            box = np.array([o["x"] - o["w"] / 2, o["y"] - o["h"] / 2,
                            o["x"] + o["w"] / 2, o["y"] + o["h"] / 2])
            if box[2] < 0 or box[0] > W_IMG:
                continue
            g.append((o["id"], box))
            # how much of the box is hidden behind the pillar?
            overlap = max(0, min(box[2], PILLAR[1]) - max(box[0], PILLAR[0]))
            hidden = overlap / (box[2] - box[0])
            if hidden > 0.7:
                # fully occluded: no detection
                continue
            score = rng.uniform(0.6, 0.95) if hidden < 0.3 else rng.uniform(0.2, 0.45)
            if rng.random() < 0.08:
                continue                                       # random miss
            c = np.r_[(box[:2] + box[2:]) / 2 + rng.normal(0, 3, 2)]
            wh = (box[2:] - box[:2]) * (1 + rng.normal(0, 0.04, 2))
            d.append(np.r_[c - wh / 2, c + wh / 2, score])
        for _ in range(rng.poisson(1.5)):                      # false positives (clutter)
            c = rng.uniform([0, 100], [W_IMG, 650]); wh = rng.uniform([30, 60], [90, 160])
            score = rng.uniform(0.5, 0.75) if rng.random() < 0.2 else rng.uniform(0.1, 0.5)
            d.append(np.r_[c - wh / 2, c + wh / 2, score])
        gt.append(g); dets.append(np.array(d).reshape(-1, 5))
    return gt, dets

objects = make_objects()
gt, dets = simulate(objects)
print(f"{len(objects)} objects, {sum(len(g) for g in gt)} ground-truth boxes, "
      f"{sum(len(d) for d in dets)} detections in {N_FRAMES} frames")

## IoU and Hungarian association

In [ ]:
from scipy.optimize import linear_sum_assignment

def iou_matrix(a, b):
    """IoU between every box in a (N x 4) and every box in b (M x 4), format x1 y1 x2 y2."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter)

def associate(track_boxes, det_boxes, iou_min=0.3):
    """Hungarian matching on 1 - IoU; pairs below iou_min are rejected (gating)."""
    iou = iou_matrix(track_boxes, det_boxes)
    if iou.size == 0:
        return [], list(range(len(track_boxes))), list(range(len(det_boxes)))
    rows, cols = linear_sum_assignment(1 - iou)
    matches = [(r, c) for r, c in zip(rows, cols) if iou[r, c] >= iou_min]
    mt = {r for r, _ in matches}; md = {c for _, c in matches}
    return (matches, [r for r in range(len(track_boxes)) if r not in mt],
            [c for c in range(len(det_boxes)) if c not in md])

a = np.array([[0, 0, 10, 10]])
b = np.array([[5, 0, 15, 10], [20, 20, 30, 30]])
print(iou_matrix(a, b).round(3))

## SORT from scratch

In [ ]:
def box_to_z(b):                    # x1 y1 x2 y2 -> cx cy w h
    return np.r_[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, b[2] - b[0], b[3] - b[1]]

def x_to_box(x):                    # state -> x1 y1 x2 y2
    cx, cy, w, h = x[:4]
    return np.r_[cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]

class Track:
    """One object: a Kalman filter on [cx, cy, w, h, vcx, vcy, vw, vh] plus bookkeeping."""
    _next_id = 1
    def __init__(self, box):
        F = np.eye(8); F[:4, 4:] = np.eye(4)                       # dt = 1 frame
        H = np.eye(4, 8)
        Q = np.diag([1, 1, 1, 1, 0.5, 0.5, 0.1, 0.1]) ** 2
        R = np.diag([3, 3, 4, 6]) ** 2
        # velocity unknown at birth
        P0 = np.diag([3, 3, 4, 6, 20, 20, 5, 5]) ** 2
        self.kf = KalmanFilter(F, H, Q, R, np.r_[box_to_z(box), np.zeros(4)], P0)
        self.id = Track._next_id; Track._next_id += 1
        self.hits, self.age_since_update = 1, 0

    def predict(self):
        self.kf.predict(); self.age_since_update += 1
        return x_to_box(self.kf.x)

    def update(self, box):
        self.kf.update(box_to_z(box)); self.hits += 1; self.age_since_update = 0

class SORT:
    def __init__(self, iou_min=0.3, max_age=30, min_hits=3, score_min=0.5, use_kf=True):
        self.tracks, self.iou_min, self.max_age = [], iou_min, max_age
        self.min_hits, self.score_min, self.use_kf = min_hits, score_min, use_kf

    def step(self, det):
        det = det[det[:, 4] >= self.score_min]
        if self.use_kf:                                            # 1. predict
            pred = np.array([t.predict() for t in self.tracks]).reshape(-1, 4)
        # "no motion model": last box
        else:
            for t in self.tracks: t.age_since_update += 1
            pred = np.array([x_to_box(t.kf.x) for t in self.tracks]).reshape(-1, 4)
        matches, um_t, um_d = associate(pred, det[:, :4], self.iou_min)   # 2. match
        for ti, di in matches:                                     # 3. update
            if self.use_kf:
                self.tracks[ti].update(det[di, :4])
            else:
                t = self.tracks[ti]; t.kf.x[:4] = box_to_z(det[di, :4])
                t.hits += 1; t.age_since_update = 0
        for di in um_d:                                            # 4. birth
            self.tracks.append(Track(det[di, :4]))
        # 5. death
        self.tracks = [t for t in self.tracks if t.age_since_update <= self.max_age]
        return [(t.id, x_to_box(t.kf.x)) for t in self.tracks
                if t.age_since_update == 0 and t.hits >= self.min_hits]

## ByteTrack-style association

In [ ]:
class ByteTrackLike(SORT):
    """SORT + ByteTrack's idea: low-score detections may update still-unmatched tracks."""
    def __init__(self, high=0.5, low=0.1, iou_low=0.5, **kw):
        super().__init__(score_min=high, **kw)
        self.high, self.low, self.iou_low = high, low, iou_low

    def step(self, det):
        hi = det[det[:, 4] >= self.high]
        lo = det[(det[:, 4] >= self.low) & (det[:, 4] < self.high)]
        pred = np.array([t.predict() for t in self.tracks]).reshape(-1, 4)
        # round 1: high scores
        m1, um_t, um_d = associate(pred, hi[:, :4], self.iou_min)
        for ti, di in m1:
            self.tracks[ti].update(hi[di, :4])
        # round 2: low scores
        m2, _, _ = associate(pred[um_t], lo[:, :4], self.iou_low)
        for k, di in m2:
            self.tracks[um_t[k]].update(lo[di, :4])
        # births only from high
        for di in um_d:
            self.tracks.append(Track(hi[di, :4]))
        self.tracks = [t for t in self.tracks if t.age_since_update <= self.max_age]
        return [(t.id, x_to_box(t.kf.x)) for t in self.tracks
                if t.age_since_update == 0 and t.hits >= self.min_hits]

## Worked example 5 · MOTA, IDF1 and five trackers

In [ ]:
def evaluate(gt, results, iou_min=0.5):
    """CLEAR-MOT (MOTA, ID switches) and IDF1 for per-frame lists of (id, box)."""
    fn = fp = idsw = n_gt = 0
    last = {}                                  # gt id -> track id it was last matched to
    # (gt id, track id) -> frames matched (for IDF1)
    pair_count = {}
    n_trk = 0
    for g, r in zip(gt, results):
        n_gt += len(g); n_trk += len(r)
        if len(g) and len(r):
            gb = np.array([b for _, b in g]); rb = np.array([b for _, b in r])
            iou = iou_matrix(gb, rb)
            rows, cols = linear_sum_assignment(-iou)
            pairs = [(i, j) for i, j in zip(rows, cols) if iou[i, j] >= iou_min]
        else:
            pairs = []
        for i, j in pairs:
            gid, tid = g[i][0], r[j][0]
            if gid in last and last[gid] != tid:
                idsw += 1                      # same object, new track id: identity switch
            last[gid] = tid
            pair_count[(gid, tid)] = pair_count.get((gid, tid), 0) + 1
        fn += len(g) - len(pairs); fp += len(r) - len(pairs)
    mota = 1 - (fn + fp + idsw) / n_gt
    # IDF1: one-to-one assignment of gt ids to track ids over the whole sequence
    gids = sorted({k[0] for k in pair_count}); tids = sorted({k[1] for k in pair_count})
    C = np.zeros((len(gids), len(tids)))
    for (gid, tid), c in pair_count.items():
        C[gids.index(gid), tids.index(tid)] = c
    rows, cols = linear_sum_assignment(-C)
    idtp = C[rows, cols].sum()
    idf1 = 2 * idtp / (n_gt + n_trk)
    return dict(MOTA=mota, IDF1=idf1, IDSW=idsw, FP=fp, FN=fn)

def run(tracker, dets):
    Track._next_id = 1
    return [tracker.step(d) for d in dets]

trackers = [("IoU only, no Kalman", SORT(use_kf=False)),
            ("SORT, max_age = 1", SORT(max_age=1)),
            ("SORT, min_hits = 1", SORT(min_hits=1)),
            ("SORT", SORT()),
            ("ByteTrack-like", ByteTrackLike())]
RESULTS = {}
for name, trk in trackers:
    m = RESULTS[name] = evaluate(gt, run(trk, dets))
    print(f"{name:20s} MOTA {m['MOTA']:.3f}  IDF1 {m['IDF1']:.3f}  "
          f"ID switches {m['IDSW']:3d}  FP {m['FP']:4d}  FN {m['FN']:5d}")

In [ ]:
def show_tracks(results, title):
    plt.figure(figsize=(8, 4))
    for fr, r in enumerate(results):
        for tid, b in r:
            plt.plot(fr, (b[0] + b[2]) / 2, ".", ms=2, color=plt.cm.tab20(tid % 20))
    plt.axhspan(*PILLAR, color="lightgrey"); plt.xlabel("frame"); plt.ylabel("box centre u")
    plt.title(title); plt.show()

show_tracks(run(SORT(use_kf=False), dets), "IoU only (no Kalman)")
show_tracks(run(SORT(), dets), "SORT")

## Worked example 6 · Moving camera and motion compensation

In [ ]:
# drone/handheld camera: the whole image shifts by a pan of up to ~25 px per frame
pan = np.cumsum(25 * np.sin(np.arange(N_FRAMES) / 12.0))            # accumulated shift in x

def shift(boxes, dx):
    out = boxes.copy(); out[:, [0, 2]] += dx
    return out

gt_cam = [[(i, b + np.r_[pan[f], 0, pan[f], 0]) for i, b in g] for f, g in enumerate(gt)]
dets_cam = [np.c_[shift(d[:, :4], pan[f]), d[:, 4]] for f, d in enumerate(dets)]

class CompensatedSORT(SORT):
    """Before predicting, move every track by the camera motion between frames."""
    def step_with_motion(self, det, dx):
        for t in self.tracks:
            # in practice dx (or a homography) comes from
            t.kf.x[0] += dx
        return self.step(det)                   # feature matching / ECC on the background

def run_comp(dets, pan):
    Track._next_id = 1
    trk = CompensatedSORT()
    d_pan = np.diff(np.r_[0, pan])
    return [trk.step_with_motion(d, dx) for d, dx in zip(dets, d_pan)]

m_plain = evaluate(gt_cam, run(SORT(), dets_cam))
m_comp = evaluate(gt_cam, run_comp(dets_cam, pan))
for label, mm in [("plain SORT", m_plain), ("motion-compensated", m_comp)]:
    print(f"moving camera, {label:18s}: MOTA {mm['MOTA']:.3f}  IDF1 {mm['IDF1']:.3f}  "
          f"ID switches {mm['IDSW']}")